In [1]:
from typing import Callable, Iterable, Type, TypeVar, cast, Literal
from pathlib import Path
import re

import spacy
from pydantic import BaseModel, Field, model_validator
from spacy import Vocab
from spacy.tokens import Doc


ATTRIBUTE_TYPE = TypeVar("ATTRIBUTE_TYPE")

TAG_RE = re.compile(r"^[A-Za-z]+(\d+)?((\.\d+)+)?")
PUNCT_RE = re.compile(r"^PUNCT")
POSITIVE_MARKERS_RE = re.compile(r"\++")
NEGATIVE_MARKERS_RE = re.compile(r"\-+")


def get_spacy_attribute(spacy_object: object,
                        attribute_name: str,
                        expected_type: Type[ATTRIBUTE_TYPE]) -> ATTRIBUTE_TYPE:
    """
    Gets an attribute from a spaCy object, whereby the attribute name can be
    prefixed with "_." to get a custom attribute. This function does not support
    nested attributes, e.g. "_._.CUSTOM_ATTRIBUTE".

    Args:
        spacy_object: The spaCy object to get the attribute from.
        attribute_name: The name of the attribute to get. If the attribute name starts with "_.",
            it is assumed to be a custom attribute.
        expected_type: The type that the attribute is expected to be.

    Returns:
        The value of the attribute.

    Raises:
        ValueError: If the attribute is not of the expected type.
    """
    spacy_custom_attribute = False
    if attribute_name[:2] == "_.":
        spacy_custom_attribute = True
        attribute_name = attribute_name[2:]
    
    attribute_value = None
    if spacy_custom_attribute:
        attribute_value = getattr(getattr(spacy_object, "_"), attribute_name)
    else:
        attribute_value = getattr(spacy_object, attribute_name)
    
    if not isinstance(attribute_value, expected_type):
        raise ValueError(f"Expected {attribute_name} to be of type {expected_type}, "
                            f"but got {type(attribute_value)}")
    return attribute_value

class USASTag(BaseModel):
    """
    Represents all of the properties associated with a USAS tag.

    Attributes:
        tag: The USAS tag.
        number_positive_markers: The number of positive markers.
        number_negative_markers: The number of negative markers.
        rarity_marker_1: True if the USAS tag contains the rarity marker %.
        rarity_marker_2: True if the USAS tag contains the rarity marker @.
        female: True if the USAS tag contains the female marker denoted by `f`.
        male: True if the USAS tag contains the male marker denoted by `m`.
        antecedents: True if the USAS tag contains the antecedents marker denoted by `c`.
        neuter: True if the USAS tag contains the neuter marker denoted by `n`.
        idiom: False, currently not supported and therefore is always False.
    """

    tag: str = Field(title="USAS Tag", description="USAS Tag", examples=["A1.1.1"])
    number_positive_markers: int = Field(
        0,
        title="Positive Markers",
        description="Number of positive markers.",
        examples=[0, 1, 2, 3],
    )
    number_negative_markers: int = Field(
        0,
        title="Negative Markers",
        description="Number of negative markers.",
        examples=[0, 1, 2, 3],
    )
    rarity_marker_1: bool = Field(
        False, title="Rare Marker 1", description="Rarity marker 1 indicated by %"
    )
    rarity_marker_2: bool = Field(
        False, title="Rare Marker 2", description="Rarity marker 2 indicated by @"
    )
    female: bool = Field(False, title="Female", description="Female")
    male: bool = Field(False, title="Male", description="Male")
    antecedents: bool = Field(
        False,
        title="Antecedents",
        description="Potential antecedents of conceptual anaphors (neutral for number)",
    )
    neuter: bool = Field(False, title="Neuter", description="Neuter")
    idiom: Literal[False] = Field(False, title="Idiom",
                                  description="Is it an idiom, currently not supported and is always False.")


class USASTagGroup(BaseModel):
    """
    Represents a grouping of one or more USAS tags that are associated to a
    token.

    Attributes:
        tags: A list of USAS tags that are associated to a token. This grouping
            of USAS tags is a way of representing multi tag membership.
    """

    _tags_description = (
        "A grouping of one or more USAS tags whereby if more "
        "than one exists then the word is an equal member of "
        "all semantic tags/categories"
    )
    _tags_examples = [
        [USASTag(tag="A1.1.1")],
        [
            USASTag(tag="E2", number_negative_markers=1),
            USASTag(tag="S7.1", number_positive_markers=1),
        ],
    ]
    tags: list[USASTag] = Field(
        title="USAS Tags", description=_tags_description, examples=_tags_examples
    )


def get_all_mwe_token_indexes(mwe_index_slices: list[tuple[int, int]]) -> frozenset[int]:
    """
    Given a list of tuples that represent the start and end indexes of a
    Multi Word Expression (MWE), it returns a frozenset of all the token indexes
    that are part of the MWE. If the MWE is a single token then it returns a
    frozenset of length 1 which is start index.

    Args:
        mwe_index_slices: A list of tuples that represent the start and end indexes of a
            Multi Word Expression (MWE).

    Returns:
        A frozenset of all the token indexes that are part of the MWE, even if
            the MWE is a single token.
    """
    all_mwe_token_indexes = set()
    for mwe_index_range in mwe_index_slices:
        all_mwe_token_indexes.update(range(*mwe_index_range))
    return frozenset(all_mwe_token_indexes)


def parse_usas_token_group(usas_tag_group_text: str) -> list[USASTagGroup]:
    r"""
    Given a the string that represents the USAS tags whereby each USAS tag is
    separated by whitespace it is converted into a structured format.

    This whitespace separation of USAS tags is the format that is produced by the
    original C version of the USAS tagger when it outputs USAS tags for a given
    token or meaningful word unit like a Multi Word Expression (MWE).

    The whitespace separation can be one or more spaces, i.e. `    ` or ` `

    A USAS tag can be also be `PUNCT` which represents punctuation.

    Complex examples of `usas_tag_group_text`:
    `L1 E3- O4.2- X5.2+ A6.2- A1.7- A7- W3 L2 F1 S1.2.4- Z2 Z2/S2mf Z3 O4.3 G1.2 G1.2/S2mf`

    Args:
        usas_tag_group_text: The string that represents the USAS tags
            produced by the USAS tagger for one token.
    Returns:
        list[USASTagGroup]: structured format of the USAS tags.
    Raises:
        ValueError: If the USAS tags within the given text cannot be parsed
            as a USAS tag, whereby each USAS tag after whitespace and `/` split
            should match the following regex: `[A-Z](\d+)((\.\d+)+)?` or `PUNCT`.
    """

    def parse_usas_tag(usas_tag_text: str) -> USASTag:
        r"""
        Given a single USAS tag text, e.g. `X5.2+` it is converted into
        a structured format.

        Note: a USAS tag text should not contain a `/`,
        e.g. `G1.2/S2mf` as this contains two USAS tags that represent
        a combined semantic meaning of a token.

        Args:
            usas_tag_text: Single USAS tag text
        Returns:
            USASTag: A structured format of the USAS tag.
        Raises:
            ValueError: If it cannot match the given text with the USAS tag
                regex, which is `[A-Z](\d+)((\.\d+)+)?` or `PUNCT`.
        """

        tag_match = TAG_RE.match(usas_tag_text)
        punct_match = PUNCT_RE.match(usas_tag_text)
        tag = ""

        if tag_match:
            tag = tag_match.group()
            usas_tag_text = TAG_RE.sub("", usas_tag_text)
        elif punct_match:
            tag = punct_match.group()
            usas_tag_text = PUNCT_RE.sub("", usas_tag_text)
        else:
            raise ValueError(
                f"Cannot find the tag for this USAS tag text: {usas_tag_text}"
            )

        number_positive_markers = 0
        positive_marker_match = POSITIVE_MARKERS_RE.search(usas_tag_text)
        if positive_marker_match:
            number_positive_markers = len(positive_marker_match.group())
            usas_tag_text = POSITIVE_MARKERS_RE.sub("", usas_tag_text)

        number_negative_markers = 0
        negative_marker_match = NEGATIVE_MARKERS_RE.search(usas_tag_text)
        if negative_marker_match:
            number_negative_markers = len(negative_marker_match.group())
            usas_tag_text = NEGATIVE_MARKERS_RE.sub("", usas_tag_text)

        is_male = False
        if "m" in usas_tag_text:
            is_male = True

        is_female = False
        if "f" in usas_tag_text:
            is_female = True

        contain_rare_marker_1 = False
        if "%" in usas_tag_text:
            contain_rare_marker_1 = True

        contain_rare_marker_2 = False
        if "@" in usas_tag_text:
            contain_rare_marker_2 = True

        contains_antecedent = False
        if "c" in usas_tag_text:
            contains_antecedent = True

        contains_neuter = False
        if "n" in usas_tag_text:
            contains_neuter = True

        # Currently do not support finding idioms
        is_idiom = False

        return USASTag(
            tag=tag,
            male=is_male,
            female=is_female,
            rarity_marker_1=contain_rare_marker_1,
            rarity_marker_2=contain_rare_marker_2,
            number_positive_markers=number_positive_markers,
            number_negative_markers=number_negative_markers,
            antecedents=contains_antecedent,
            neuter=contains_neuter,
            idiom=is_idiom,
        )

    token_usas_tags: list[USASTagGroup] = []

    for usas_tag_group in re.findall(r"\S+", usas_tag_group_text):
        usas_tags: list[USASTag] = []
        for usas_tag_text in usas_tag_group.split("/"):
            usas_tags.append(parse_usas_tag(usas_tag_text))
        token_usas_tags.append(USASTagGroup(tags=usas_tags))

    return token_usas_tags


def load_usas_mapper(usas_tag_descriptions_file: Path | None,
                     tags_to_filter_out: set[str] | None
                     ) -> dict[str, str]:
    """
    Returns a dictionary of USAS tags and their descriptions.

    Args:
        usas_tag_descriptions_file: The path to the YAML file that
            contains the USAS tags and their descriptions. If None then the
            function will use the USAS tags and description file that is located
            within the package at `usas_csv_auto_labeling/data/usas/usas_mapper.yaml`.
        tags_to_filter_out: A set of USAS tags to filter out.

    Returns:
        dict[str, str]: A dictionary of USAS tags and their descriptions.
    
    Raises:
        FileNotFoundError: If the `usas_tag_descriptions_file` is not found.
        ValueError: If the `usas_tag_descriptions_file` is not a file.
    """

    def _get_usas_tag_descriptions(usas_tag_name: str,
                                   usas_tag_dict: dict[str, Any],
                                   collected_tag_descriptions: dict[str, str]
                                   ) -> dict[str, str]:
        """
        A recursive function that loops through the `usas_tag_dict` and returns a
        dictionary of the USAS tag and as a value it's description. Each USAS tag
        that is found is added to the `collected_tag_descriptions` dictionary.
        Once all the USAS tags are found, the `collected_tag_descriptions` is
        returned.

        The description is made of the USAS tag title and description in the
        following format: `title: <title> description: <description>`

        Args:
            usas_tag_name: The name of the USAS tag.
            usas_tag_dict: A dictionary containing the raw
                USAS tag data that is read from the YAML file.
            collected_tag_descriptions: A dictionary of all
                USAS tags and their descriptions.

        Returns:
            dict[str, str]: A dictionary of USAS tags and their descriptions.
        """
        if "title" in usas_tag_dict and "description" in usas_tag_dict:
            title_description = f"title: {usas_tag_dict['title']} description: {usas_tag_dict['description']}"
            if usas_tag_name in collected_tag_descriptions:
                raise KeyError(f"Duplicate usas tag name found: {usas_tag_name} "
                               "when reading the following data: "
                               f"{usas_tag_dict}, currently found usas tags: "
                               f"{collected_tag_descriptions}")
            collected_tag_descriptions[usas_tag_name] = title_description.strip()
        elif "title" in usas_tag_dict:
            raise KeyError("No description key found when it is expected for: "
                           f"{usas_tag_name} {usas_tag_dict}")
        elif "description" in usas_tag_dict:
            raise KeyError("No title key found when it is expected for: "
                           f"{usas_tag_name} {usas_tag_dict}")

        keys_to_ignore = set(["title", "description"])
        for child_usas_tag_name, child_usas_tag_dict in usas_tag_dict.items():
            if child_usas_tag_name not in keys_to_ignore:
                collected_tag_descriptions = _get_usas_tag_descriptions(child_usas_tag_name,
                                                                          child_usas_tag_dict,
                                                                                        collected_tag_descriptions)
        return collected_tag_descriptions

    usas_tag_descriptions_file_path: Path = Path()
    if usas_tag_descriptions_file is None:
        usas_tag_descriptions_file_str = str(files("usas_csv_auto_labeling").joinpath("data/usas/usas_mapper.yaml"))
        usas_tag_descriptions_file_path = Path(usas_tag_descriptions_file_str)
    else:
        usas_tag_descriptions_file_path = usas_tag_descriptions_file

    if usas_tag_descriptions_file_path.exists() is False:
        raise FileNotFoundError(f"USAS tag descriptions file not found at: "
                                f"{usas_tag_descriptions_file_path}")
    elif usas_tag_descriptions_file_path.is_file() is False:
        raise ValueError(f"USAS tag descriptions file is not a file: "
                         f"{usas_tag_descriptions_file_path}")

    usas_mapping: dict[str, str] = {}
    with usas_tag_descriptions_file_path.open("r") as usas_mapper_fp:
        usas_mapping_data = usas_mapper_fp.read()
        for high_level_usas_tag, high_level_usas_tag_dict in yaml.safe_load(usas_mapping_data).items():
            usas_mapping = _get_usas_tag_descriptions(high_level_usas_tag,
                                                      high_level_usas_tag_dict,
                                                      usas_mapping)
    if tags_to_filter_out:
        tmp_usas_mapping = {}
        for key, value in usas_mapping.items():
            if key in tags_to_filter_out:
                continue
            tmp_usas_mapping[key] = value
        usas_mapping = tmp_usas_mapping
    return usas_mapping

class TaggedText(BaseModel):
    """
    A class that represents a tagged text.

    Attributes:
        text (str): The text that was tagged.
        tokens (list[str]): The tokens of the text that was tagged.
        lemmas (list[str] | None):: The lemmas of the text that was tagged. Default None.
        pos_tags (list[str] | None): The POS tags of the text that was tagged. Default None.
        usas_tags (list[list[USASTagGroup]]): The USAS tags of the text that was tagged.
        mwe_indexes (list[set[int]]): The MWE indexes of the text that was tagged.
    """
    _usas_tags_example: list[list[USASTagGroup]] = [
        [
            USASTagGroup(tags=[USASTag(tag='M6', number_positive_markers=0, number_negative_markers=0, rarity_marker_1=False, rarity_marker_2=False, female=False, male=False, antecedents=False, neuter=False, idiom=False)]),
            USASTagGroup(tags=[USASTag(tag='Z5', number_positive_markers=0, number_negative_markers=0, rarity_marker_1=False, rarity_marker_2=False, female=False, male=False, antecedents=False, neuter=False, idiom=False)]),
            USASTagGroup(tags=[USASTag(tag='Z8', number_positive_markers=0, number_negative_markers=0, rarity_marker_1=False, rarity_marker_2=False, female=False, male=False, antecedents=False, neuter=False, idiom=False)])
        ],
        [
            USASTagGroup(tags=[USASTag(tag='A3', number_positive_markers=1, number_negative_markers=0, rarity_marker_1=False, rarity_marker_2=False, female=False, male=False, antecedents=False, neuter=False, idiom=False)]),
            USASTagGroup(tags=[USASTag(tag='Z5', number_positive_markers=0, number_negative_markers=0, rarity_marker_1=False, rarity_marker_2=False, female=False, male=False, antecedents=False, neuter=False, idiom=False)])
        ],
        [
            USASTagGroup(tags=[USASTag(tag='Z5', number_positive_markers=0, number_negative_markers=0, rarity_marker_1=False, rarity_marker_2=False, female=False, male=False, antecedents=False, neuter=False, idiom=False)])
        ],
        [
            USASTagGroup(tags=[USASTag(tag='Q3', number_positive_markers=0, number_negative_markers=0, rarity_marker_1=False, rarity_marker_2=False, female=False, male=False, antecedents=False, neuter=False, idiom=False)]),
            USASTagGroup(tags=[USASTag(tag='G2.1', number_positive_markers=0, number_negative_markers=0, rarity_marker_1=False, rarity_marker_2=False, female=False, male=False, antecedents=False, neuter=False, idiom=False)])
        ],
        [
            USASTagGroup(tags=[USASTag(tag='PUNCT', number_positive_markers=0, number_negative_markers=0, rarity_marker_1=False, rarity_marker_2=False, female=False, male=False, antecedents=False, neuter=False, idiom=False)])
        ]
    ]
    _mwe_indexes_description = """
    Multi Word Expression (MWE) indexes for each token in the tagged text. If a token
    is a MWE then it is assigned an index starting from 1 if a token is part of more than
    one MWE then it is assigned two MWE indexes, e.g. `set([1,2])` means that the token for
    that index is part of MWE 1 and 2. An empty set represents a token that is not part
    of any MWE. TO NOTE: USAS historically does not support overlapping MWEs.
    """
    text: str = Field(title="Text", description="The text that was tagged", examples=["This is a sentence.", ""])
    tokens: list[str] = Field(title="Tokens", description="The tokens of the text that was tagged", examples=[["This", "is", "a", "sentence", "."], []])
    lemmas: list[str] | None = Field(title="Lemmas", description="The lemmas of the text that was tagged", examples=[["this", "be", "a", "sentence", "."], None], default=None)
    pos_tags: list[str] | None = Field(title="POS Tags", description="The POS tags of the text that was tagged", examples=[["PRON", "AUX", "DET", "NOUN", "PUNCT"], None], default=None)
    usas_tags: list[list[USASTagGroup]] = Field(title="USAS Tags", description="The USAS tags of the text that was tagged", examples=[_usas_tags_example, []])
    mwe_indexes: list[set[int]] = Field(title="MWE indexes", description=_mwe_indexes_description, examples=[[set(), set(), set([1]), set([1]), set()], []])

    @model_validator(mode='after')
    def check_lists_match(self) -> "TaggedText":
        """
        Checks that the length of the tokens, lemmas, POS tags, USAS tags, and MWE indexes
        are all the same if they are not None. If they are not the same, raises a ValueError.

        Returns:
            The TaggedText object
        Raises:
            ValueError: If the length of the tokens, lemmas, POS tags, USAS tags, and MWE indexes are not the same
        """
        number_tokens = len(self.tokens)
        if self.lemmas is not None and number_tokens != len(self.lemmas):
            raise ValueError(f"The number of tokens: {number_tokens} and "
                             f"lemmas must be the same: {len(self.lemmas)}")
        if self.pos_tags is not None and number_tokens != len(self.pos_tags):
            raise ValueError(f"The number of tokens: {number_tokens} "
                             f"and POS tags must be the same: {len(self.pos_tags)}")
        if number_tokens != len(self.usas_tags):
            raise ValueError(f"The number of tokens: {number_tokens} and "
                             f"USAS tags must be the same: {len(self.usas_tags)}")
        if number_tokens != len(self.mwe_indexes):
            raise ValueError(f"The number of tokens: {number_tokens} and "
                             f"MWE indexes must be the same: {len(self.mwe_indexes)}")
        return self

def process_text(text_to_process: str | Doc,
                 spacy_tagger: spacy.Language,
                 lemma_token_extension: str | None = None, # lemma_
                 pos_token_extension: str | None = None, # pos_
                 token_text_extension: str = "text",
                 usas_token_extension: str = "_.pymusas_tags",
                 mwe_token_extension: str = "_.pymusas_mwe_indexes") -> TaggedText:
    """
    Processes a text by tagging it with a spaCy pipeline and returning
    a TaggedText object.

    Args:
        text_to_process: The text to process.
        spacy_tagger: The spaCy pipeline to use for tagging.
        lemma_token_extension: The name of the attribute to get the lemma of a token
            from the spacy Token object. If not provided, the lemma will not be extracted.
        pos_token_extension: The name of the attribute to get the POS tag of a token
            from the spacy Token object. If not provided, the POS tag will not be extracted.
        token_text_extension: The name of the attribute to get the text of a token
            from the spacy Token object.
        usas_token_extension: The name of the custom attribute to get the USAS tags of a token
            from the spacy Token object.
        mwe_token_extension: The name of the custom attribute to get the MWE indexes of a token
            from the spacy Token object.

    Returns:
        A TaggedText object, which contains the text, tokens, lemmas, POS tags, USAS tags, and MWE indexes.
    """
    tagged_text: Doc = spacy_tagger(text_to_process)

    tokens: list[str] = []
    lemmas: list[str] | None = []
    pos_tags: list[str] | None = []
    usas_tag_groups: list[list[USASTagGroup]] = []
    mwe_indexes: list[set[int]] = []
    all_mwe_token_indexes_with_min_value: set[tuple[frozenset[int], int]] = set()

    for token in tagged_text:
        token_text = get_spacy_attribute(token, token_text_extension, str)
        tokens.append(token_text)
        
        if lemma_token_extension is not None:
            lemma = get_spacy_attribute(token, lemma_token_extension, str)
            lemmas.append(lemma)
        
        if pos_token_extension is not None:
            pos_tag = get_spacy_attribute(token, pos_token_extension, str)
            pos_tags.append(pos_tag)

        usas_tags = cast(list[str],
                                    get_spacy_attribute(token, usas_token_extension, list))
        token_usas_tag_groups = parse_usas_token_group(" ".join(usas_tags))
        usas_tag_groups.append(token_usas_tag_groups)

        
        
        token_mwe_indexes_range = cast(list[tuple[int, int]],
                                                                get_spacy_attribute(token, mwe_token_extension, list))
        
        token_mwe_indexes = get_all_mwe_token_indexes(token_mwe_indexes_range)
        if len(token_mwe_indexes) > 1:
            all_mwe_token_indexes_with_min_value.add((token_mwe_indexes, min(token_mwe_indexes)))
        mwe_indexes.append(set())

    # Computationally mwe indexes in token order has increased the complexity
    # of the code and runtime, but it should make it easier for the annotators
    # to read.
    sorted_all_mwe_token_indexes_with_min_value = sorted(all_mwe_token_indexes_with_min_value, key=lambda x: x[1])
    for mwe_index_value, mwe_index_with_min_value in enumerate(sorted_all_mwe_token_indexes_with_min_value, start=1):
        for mwe_index in mwe_index_with_min_value[0]:
            mwe_indexes[mwe_index].add(mwe_index_value)

    if lemma_token_extension is None:
        lemmas = None
    
    if pos_token_extension is None:
        pos_tags = None
    
    tagged_text_text_string = text_to_process
    if isinstance(tagged_text_text_string, Doc):
        if "original_text" in tagged_text_text_string.user_data:
            original_text = tagged_text_text_string.user_data.get("original_text")
            assert isinstance(original_text, str)
            tagged_text_text_string = original_text
        else:
            tagged_text_text_string = ""
    
    return TaggedText(
        text=tagged_text_text_string,
        tokens=tokens,
        lemmas=lemmas,
        pos_tags=pos_tags,
        usas_tags=usas_tag_groups,
        mwe_indexes=mwe_indexes
    )

    return tagged_text

In [18]:
from sympy.multipledispatch.dispatcher import source
from pymusas.lexicon_collection import LexiconCollection, MWELexiconCollection
from pymusas.pos_mapper import UPOS_TO_USAS_CORE, USAS_CORE_TO_UPOS
from pymusas.rankers.lexicon_entry import ContextualRuleBasedRanker
from pymusas.spacy_api.taggers.hybrid import HybridTagger
from pymusas.taggers.rules.mwe import MWERule
from pymusas.taggers.rules.single_word import SingleWordRule

def get_english_rule_based_tagger() -> spacy.Language:
    nlp = spacy.load('en_core_web_trf', exclude=['parser', 'ner'])
    english_tagger_pipeline = spacy.load('en_dual_none_contextual_none')
    nlp.add_pipe('pymusas_rule_based_tagger', source=english_tagger_pipeline)
    return nlp


def get_english_hybrid_tagger() -> spacy.Language:

    hybrid_nlp = spacy.load('en_core_web_trf', exclude=['parser', 'ner'])
    # URLS to the English single and MWE lexicons
    english_single_lexicon_url = ('https://raw.githubusercontent.com/UCREL/Multilingual-USAS/'
                                  '2cc9966a3bdcc84bc204d16bdf4318fc28495016/'
                                  'English/semantic_lexicon_en.tsv')
    english_mwe_lexicon_url = ('https://raw.githubusercontent.com/UCREL/Multilingual-USAS/'
                               '2cc9966a3bdcc84bc204d16bdf4318fc28495016/'
                               'English/mwe-en.tsv')
    lexicon_lookup = LexiconCollection.from_tsv(english_single_lexicon_url, include_pos=True)
    lemma_lexicon_lookup = LexiconCollection.from_tsv(english_single_lexicon_url, include_pos=False)
    mwe_lexicon_lookup = MWELexiconCollection.from_tsv(english_mwe_lexicon_url)
    # The rules that use the lexicons
    single_word_rule = SingleWordRule(lexicon_lookup, lemma_lexicon_lookup)
    mwe_word_rule = MWERule(mwe_lexicon_lookup)
    word_rules = [single_word_rule, mwe_word_rule]
    # The ranker that determines which rule should be used/applied
    ranker_arguments = ContextualRuleBasedRanker.get_construction_arguments(word_rules)
    ranker = ContextualRuleBasedRanker(*ranker_arguments)
    # POS that indicate a Punctuation and Numeric value
    default_punctuation_tags = list(['PUNCT'])
    default_number_tags = list(['NUM'])

    tagger = cast(HybridTagger, hybrid_nlp.add_pipe('pymusas_hybrid_tagger', config={"top_n": 5}))

    tagger.initialize(rules=word_rules,
                      ranker=ranker,
                      default_punctuation_tags=default_punctuation_tags,
                      default_number_tags=default_number_tags,
                      pretrained_model_name_or_path="ucrelnlp/PyMUSAS-Neural-English-Base-BEM")
    return hybrid_nlp

def get_english_neural_tagger() -> spacy.Language:

    nlp = spacy.blank("en")
    neural_nlp = spacy.load('en_none_none_none_englishbasebem',
                                      config={"components.pymusas_neural_tagger.top_n": 5})
    nlp.add_pipe("pymusas_neural_tagger", source=neural_nlp)

    return nlp


In [19]:
english_hybrid_tagger = get_english_hybrid_tagger()
english_neural_tagger = get_english_neural_tagger()
english_rule_based_tagger = get_english_rule_based_tagger()

/workspaces/USAS-Evaluation-Framework/.venv/lib/python3.13/site-packages/pymusas/lexicon_collection.py:1038: UserWarning: We do not currently support Curly Braces expressions within Multi Word Expression (MWE) lexicons and therefore any MWE template that contains a `{` or `}` will be ignored.
  warnings.warn('We do not currently support Curly Braces expressions'
/workspaces/USAS-Evaluation-Framework/.venv/lib/python3.13/site-packages/pymusas/spacy_api/utils.py:38: UserWarning: Overwritten the spaCy Token extension `pymusas_tags` which currently has the following (default, method, getter, setter):`(None, None, None, None)`. And replacing it with the following:`(None, None, None, None)`. This would only become a problem if the the two Tuples of four are different, if they are the same there is no problem.
  warnings.warn(message)
/workspaces/USAS-Evaluation-Framework/.venv/lib/python3.13/site-packages/pymusas/spacy_api/utils.py:38: UserWarning: Overwritten the spaCy Token extension `pymu

In [5]:
from huggingface_hub import snapshot_download

evaluation_data_folder_path = snapshot_download(repo_id="ucrelnlp/USAS-WSD", repo_type="dataset", revision="main")
evaluation_data_folder_path = Path(evaluation_data_folder_path) / "test"
english_data = evaluation_data_folder_path / "benedict_eng.txt"

Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 66752.85it/s]


In [6]:
from usas_evaluation_framework.parsers.benedict import EnglishBenedict
from usas_evaluation_framework.data_utils import load_usas_mapper

relevant_usas_labels = set(load_usas_mapper(None, ["Z99"]).keys())
english_dataset = EnglishBenedict.parse(english_data, label_validation=relevant_usas_labels, label_filter=set(["PUNCT", "Z99"]))

In [7]:
from usas_evaluation_framework.dataset import EvaluationDataset, EvaluationTexts, TextLevel
from usas_evaluation_framework.metrics import top_n_accuracy

In [28]:
english_tagger = english_rule_based_tagger
all_eval_texts = []
for text in english_dataset.texts:
    tokens = text.tokens
    text = text.text
    doc = Doc(english_tagger.vocab, words=tokens)
    result = process_text(doc, english_tagger)
    token_usas_tags = []

    for token_usas_tag_groups in result.usas_tags:
        _token_usas_tags = []
        for token_usas_tag_group in token_usas_tag_groups:
            usas_tag_list = []
            for tag in token_usas_tag_group.tags:
                usas_tag_list.append(tag.tag)
            _token_usas_tags.append("/".join(usas_tag_list))
        token_usas_tags.append(_token_usas_tags)
    
    all_eval_texts.append(EvaluationTexts(text=text, tokens=result.tokens, lemmas=None, pos_tags=None, semantic_tags=token_usas_tags, mwe_indexes=result.mwe_indexes))
pred_eval_dataset = EvaluationDataset(name="English Pred", text_level=TextLevel.sentence, labels_removed=["Z99"], texts=all_eval_texts)
    

In [29]:
top_n_accuracy.top_n_accuracy(english_dataset, pred_eval_dataset, n=1, average="micro")

0.7234717416378316

In [30]:
top_n_accuracy.top_n_accuracy(english_dataset, pred_eval_dataset, n=5, average="micro")

0.817762399077278

In [13]:
top_n_accuracy.top_n_accuracy(english_dataset, pred_eval_dataset, n=1, average="macro")

0.45366876039417964

In [14]:
top_n_accuracy.top_n_accuracy(english_dataset, pred_eval_dataset, n=5, average="macro")

0.6911169346734058

In [26]:
top_n_accuracy.top_n_accuracy(english_dataset, pred_eval_dataset, n=1, average="micro")

0.7246251441753172

In [27]:
top_n_accuracy.top_n_accuracy(english_dataset, pred_eval_dataset, n=5, average="micro")

0.8189158016147635